# Phase 0 — Real ML pipeline proof
Runs F5-TTS → LivePortrait → MuseTalk in two isolated conda envs and produces one talking-head MP4.

**Settings → Accelerator: GPU T4 x2, Internet: ON.** Run cells top to bottom.

In [ ]:
# 1. Clone the repo (gets the proof scripts)
import os, subprocess
os.chdir('/kaggle/working')
if not os.path.exists('Ai-Avatar'):
    subprocess.run(['git','clone','https://github.com/Yashvant203/Ai-Avatar.git'], check=True)
print('repo ready')

In [ ]:
# 2. Build env-A (F5-TTS + LivePortrait). ~10-15 min.
!bash /kaggle/working/Ai-Avatar/ml_models/proof/setup_env_a.sh

In [ ]:
# 3. Build env-B (MuseTalk + mmcv). ~15-25 min. The riskiest install.
!bash /kaggle/working/Ai-Avatar/ml_models/proof/setup_env_b.sh

In [ ]:
# 4. F5-TTS: clone a voice from F5's bundled reference, synthesize a test line.
import subprocess
M = ['bash','Ai-Avatar/ml_models/proof/mrun.sh','envA']
REF = subprocess.check_output(M + ['python','-c',
    "from importlib.resources import files; print(files('f5_tts').joinpath('infer/examples/basic/basic_ref_en.wav'))"]).decode().strip()
subprocess.run(M + ['python',
    'Ai-Avatar/ml_models/proof/run_f5tts.py','--ref',REF,'--ref-text','',
    '--gen-text','Hello, this is a test of my cloned voice on a free GPU.',
    '--out','/kaggle/working/speech.wav'], check=True)
print('speech.wav written')

In [ ]:
# 5. Make the driving clip match the speech length, then animate with LivePortrait.
import subprocess
dur = float(subprocess.check_output(['ffprobe','-v','error','-show_entries',
    'format=duration','-of','default=noprint_wrappers=1:nokey=1',
    '/kaggle/working/speech.wav']).decode().strip())
print('speech duration', round(dur,2), 's')
drv_src = '/kaggle/working/LivePortrait/assets/examples/driving/d0.mp4'
# loop the idle driving clip to >= speech duration
subprocess.run(['ffmpeg','-y','-stream_loop','-1','-i',drv_src,'-t',f'{dur:.2f}',
    '-r','25','/kaggle/working/driving_loop.mp4'], check=True)
subprocess.run(['bash','Ai-Avatar/ml_models/proof/run_liveportrait.sh',
    '/kaggle/working/LivePortrait/assets/examples/source/s9.jpg',
    '/kaggle/working/driving_loop.mp4','/kaggle/working/lp_out'], check=True)
import glob; animated = sorted(glob.glob('/kaggle/working/lp_out/*.mp4'))[-1]; print('animated:', animated)

In [ ]:
# 6. Normalize to MuseTalk inputs: 25fps video + 16kHz mono audio.
import glob, subprocess
animated = sorted(glob.glob('/kaggle/working/lp_out/*.mp4'))[-1]
subprocess.run(['ffmpeg','-y','-i',animated,'-r','25','/kaggle/working/animated_25.mp4'], check=True)
subprocess.run(['ffmpeg','-y','-i','/kaggle/working/speech.wav','-ar','16000','-ac','1','/kaggle/working/speech_16k.wav'], check=True)
print('normalized inputs ready')

In [ ]:
# 7. MuseTalk lip-sync (env-B), from the MuseTalk repo root.
import subprocess
subprocess.run(['bash','/kaggle/working/Ai-Avatar/ml_models/proof/mrun.sh','envB','python',
    '/kaggle/working/Ai-Avatar/ml_models/proof/run_musetalk.py',
    '--video','/kaggle/working/animated_25.mp4',
    '--audio','/kaggle/working/speech_16k.wav',
    '--result-dir','/kaggle/working/mt_out'], cwd='/kaggle/working/MuseTalk', check=True)
import glob; print('musetalk out:', glob.glob('/kaggle/working/mt_out/**/*.mp4', recursive=True))

In [ ]:
# 8. Mux the cloned speech onto the lip-synced video → output.mp4, then display.
import glob, subprocess
lip = glob.glob('/kaggle/working/mt_out/**/*.mp4', recursive=True)[-1]
subprocess.run(['ffmpeg','-y','-i',lip,'-i','/kaggle/working/speech.wav',
    '-c:v','copy','-c:a','aac','-shortest','/kaggle/working/output.mp4'], check=True)
from IPython.display import Video; Video('/kaggle/working/output.mp4', embed=True, width=512)

## Done
If `output.mp4` shows the face speaking with plausible motion + lip-sync, the stack is proven.

### Use your own face + voice
Replace the bundled assets: in cell 4 set `--ref` to a ~10s clean wav of you (and `--ref-text` to its transcript); in cell 5 set the source image to your face photo. Both can be extracted from your training clip with ffmpeg.

### Cache weights
Save `LivePortrait/pretrained_weights`, `MuseTalk/models`, and `~/.cache/huggingface` as a Kaggle Dataset, then mount it next session and skip cells 2–3's downloads.